In [17]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York") 

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from TB.FixedRateBondsTB import FixedRateBondsTB

from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedStructure, UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

usts_mdp = FixedRateBondsMDP(source="USTS_WEBULL_WSJ_LIVE-RL")
ts_builder = TimeseriesBuilder()

In [24]:
start = NYC_tz.localize(datetime.datetime(2026, 3, 23, 7, 00))
end = NYC_tz.localize(datetime.datetime(2026, 3, 24, 17, 00))

q1 = UnifiedQuery(
	cusip="o2/CT2",
	value=UnifiedValue.FRB_YTM,
)
df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q1],
    routers={
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    freq="1min"
)
df

PRICING FIXED-RATE BONDS.: 100%|██████████| 839/839 [00:00<00:00, 521568.26it/s]


,o2/CT2 CURVE YTM
Date,
2026-03-23 07:00:00-04:00,-3.7
2026-03-23 07:01:00-04:00,-3.7
2026-03-23 07:02:00-04:00,-3.7
2026-03-23 07:03:00-04:00,-3.7
2026-03-23 07:04:00-04:00,-3.7
...,...
2026-03-24 16:56:00-04:00,-3.0
2026-03-24 16:57:00-04:00,-3.0
2026-03-24 16:58:00-04:00,-3.0


In [25]:
plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(df[q1.col_name()], which="left")
legend(valfmt="{:.3f}", show_date=True)